# Differential Privacy Training with AdvSecureNet

This notebook demonstrates how to train a model with differential privacy using AdvSecureNet. We'll train a ResNet-18 model on the CIFAR-10 dataset while preserving privacy using the Opacus library in the background.

## What is Differential Privacy?

Differential privacy limits the extent to which the model’s output can reveal information about any individual training example.

### Key Parameters:
- **`noise_multiplier`**: Controls the amount of noise added during training. Higher values = more privacy but potentially lower utility
- **`max_grad_norm`**: Maximum L2 norm for gradient clipping. Bounds the sensitivity of the model
- **`delta`**: Privacy parameter that bounds the probability of privacy failure
- **`kwargs`**: Additional Opacus parameters ⚠️ **Note**: When using kwargs, ensure that parameter names match exactly what Opacus expects to avoid runtime errors. Compatibility with advsecurenet is not guaranteed for all possible kwargs modifications.

## Step 1: Setup and Imports

In [ ]:
# Core imports
import torch
import gc

# AdvSecureNet imports
from advsecurenet.models.model_factory import ModelFactory
from advsecurenet.datasets.dataset_factory import DatasetFactory
from advsecurenet.dataloader.data_loader_factory import DataLoaderFactory
from advsecurenet.trainer.trainer import Trainer
from advsecurenet.shared.types.configs.preprocess_config import (
    PreprocessConfig,
    PreprocessStep,
)
from advsecurenet.shared.types.configs.device_config import DeviceConfig
from advsecurenet.shared.types.configs import TrainConfig
from advnet_common.types.configs.base import (
    CheckpointBase,
    FinalModelBase,
    OptimizationBase,
    DifferentialPrivacyBase
)
from advsecurenet.shared.types.configs.train_config import (
    ModelConfig,
    TrainingProcessConfig,
    DeviceConfig
    
)

## Step 2: Create ResNet-18 Model

We'll create a ResNet-18 model configured for CIFAR-10 (10 classes).

In [ ]:
# Create ResNet-18 model for CIFAR-10
model = ModelFactory.create_model(
    model_name="resnet18",  
    architecture={"num_classes": 10},  
    pretrained=False 
)

# FORCE the final layer to have exactly 10 classes
import torch.nn as nn

# Access the underlying model
if hasattr(model, 'model'):
    base_model = model.model
else:
    base_model = model

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
gc.collect()

# Force the final layer to have 10 classes
if hasattr(base_model, 'fc'):
    # Get the input features from current fc layer
    in_features = base_model.fc.in_features
    # Replace with new fc layer having exactly 10 classes
    base_model.fc = nn.Linear(in_features, 10)
    print(f"✅ FORCED final layer to have 10 classes (in_features={in_features})")

# Make the model compatible with Opacus by fixing in-place operations
from opacus.validators import ModuleValidator

# First, fix in-place operations manually
def fix_inplace_operations(module):
    for name, child in module.named_children():
        if isinstance(child, torch.nn.ReLU):
            child.inplace = False
        else:
            fix_inplace_operations(child)

# Apply the manual fix to the underlying model
if hasattr(model, 'model'):
    fix_inplace_operations(model.model)
else:
    fix_inplace_operations(model)

# Then apply Opacus validator fix
model = ModuleValidator.fix(model)

print("Loaded ResNet-18 model with 10 classes for CIFAR-10")
print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")

# Verify the final layer has correct number of classes
if hasattr(model, 'model') and hasattr(model.model, 'fc'):
    fc_layer = model.model.fc
elif hasattr(model, 'fc'):
    fc_layer = model.fc
else:
    fc_layer = None

if fc_layer:
    print(f"Final layer output classes: {fc_layer.out_features}")
    if fc_layer.out_features == 10:
        print("✅ Model correctly configured for CIFAR-10")
    else:
        print(f"❌ Model has {fc_layer.out_features} classes instead of 10")
        # Force fix if still wrong
        fc_layer = nn.Linear(fc_layer.in_features, 10)
        if hasattr(model, 'model'):
            model.model.fc = fc_layer
        else:
            model.fc = fc_layer
        print("🔧 FORCE FIXED: Final layer now has 10 classes")

## Step 3: Setup Data Preprocessing

Configure preprocessing transforms for CIFAR-10 dataset.

In [ ]:
# Create preprocessing configuration
preprocess_config = PreprocessConfig(
    steps=[
        PreprocessStep(name="Resize", params={"size": 32}),
        PreprocessStep(name="CenterCrop", params={"size": 32}),
        PreprocessStep(name="ToTensor"),
        PreprocessStep(
            name="ToDtype", params={"dtype": "torch.float32", "scale": True}
        ),
        PreprocessStep(
            name="Normalize",
            params={"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
        ),
    ]
)

## Step 4: Create CIFAR-10 Dataset and DataLoader

Load the CIFAR-10 dataset and create data loaders for training.

In [ ]:
# Create CIFAR-10 dataset
dataset = DatasetFactory.load_dataset(
    dataset_name="cifar10", 
    preprocessing=preprocess_config, 
    num_classes=10
)

train_data = dataset['train']
test_data = dataset['test']

print("CIFAR-10 dataset loaded")
print(f"Training samples: {len(train_data):,}".replace(",", "'"))
print(f"Test samples: {len(test_data):,}".replace(",", "'"))

In [ ]:
# Create data loaders
# Note: For differential privacy, batch size should be chosen carefully
# Smaller batches may provide better privacy but slower training
train_loader = DataLoaderFactory.create_dataloader(
    dataset=train_data, 
    batch_size=128,  # Balanced batch size for DP training
    shuffle=True,
    drop_last=True  # Important for DP: ensures consistent batch sizes
)

print("Data loader created")
print(f"Training batches: {len(train_loader)}")
print(f"Training batch size: {train_loader.batch_size}")

## Step 5: Configure Differential Privacy

This is the key step where we configure differential privacy parameters.

### Privacy Parameters Explained:
- **`noise_multiplier=1.2`**: Moderate noise level for reasonable privacy-utility trade-off
- **`max_grad_norm=1.0`**: Standard gradient clipping threshold
- **`delta=1e-5`**: Small probability of privacy failure (< 1 in 100,000)
- **`kwargs`**: Additional Opacus-specific parameters

In [ ]:
# Configure differential privacy
# IMPORTANT: When using kwargs, ensure parameter names match Opacus expectations exactly!
differential_privacy_config = DifferentialPrivacyBase(
    enable=True,
    noise_multiplier=0.5,      # Higher values = more privacy, lower utility
    max_grad_norm=1.0,         # Gradient clipping threshold
    delta=1e-5,                # Privacy failure probability
    kwargs={
        # Example additional parameters (uncomment as needed)
        # "clipping": "flat",     # Gradient clipping method: "flat" or "fast"
        # "loss_reduction": "mean"  # How to reduce loss across samples
    }
)

print("Differential Privacy Configuration:")
print(f"   • Enabled: {differential_privacy_config.enable}")
print(f"   • Noise Multiplier: {differential_privacy_config.noise_multiplier}")
print(f"   • Max Gradient Norm: {differential_privacy_config.max_grad_norm}")
print(f"   • Delta: {differential_privacy_config.delta}")
print(f"   • Additional kwargs: {differential_privacy_config.kwargs}")
print()
print("NOTE: Higher noise_multiplier = more privacy but potentially lower model performance")

## Step 6: Create Training Configuration

Configure the training process with differential privacy enabled.

In [ ]:
# Training configuration with differential privacy
config = TrainConfig(
    model_config=ModelConfig(model=model),
    training_process_config=TrainingProcessConfig(
        train_loader=train_loader,
        epochs=20,                          
        learning_rate=0.01,               
        criterion="cross_entropy",
        verbose=True,
    ),
    optimization_config=OptimizationBase(
        optimizer="sgd",               # SGD often works better with DP
        optimizer_kwargs={
            "momentum": 0.9,
            "weight_decay": 1e-4
        }
    ),
    
    #If DeviceConfig gets initalized empty it uses the first available device in the following priortity order: CUDA -> MPS -> CPU
    #If you want to manually set a device you can add the device as a string manually in the initalisation of DeviceConfig e.g. DeviceConfig(processor="cuda") or if you want to specify the GPU id (in this example GPU id is 2) DeviceConfig(processor="cuda:2")
    device_config=DeviceConfig(),      
    checkpoint_config=CheckpointBase(),
    final_model_config=FinalModelBase(
        save_final_model=True,
        save_model_path="./models",
        save_model_name="resnet18_cifar10_dp_moderate_privacy"
    ),
    differential_privacy_config=differential_privacy_config,
)

print("Training Configuration:")
print(f"   • Epochs: {config.training_process_config.epochs}")
print(f"   • Learning Rate: {config.training_process_config.learning_rate}")
print(f"   • Optimizer: {config.optimization_config.optimizer}")
print(f"   • Differential Privacy: {'ENABLED' if config.differential_privacy_config.enable else 'DISABLED'}")

## Step 7: Initialize Trainer and Start Training

Create the trainer and begin differential privacy training.

In [ ]:
# Create trainer
trainer = Trainer(config)
print(f"Device: {trainer._device}")
print("Trainer initialized with differential privacy!")
print("Starting training with privacy-preserving guarantees...\n")

# Start training
trainer.train()

print()
print("Differential privacy training test completed successfully!")

In [ ]:
# Step 8: Evaluate Model Performance on Test Set (FIXED VERSION)

def strip_opacus_prefix(state_dict, is_opacus_model=True):
    """
    Strip Opacus '_module.' prefix from state dict keys if needed.
    
    Args:
        state_dict: Model state dictionary
        is_opacus_model: Whether model was trained with Opacus (adds '_module.' prefix)
    
    Returns:
        Cleaned state dictionary
    """
    if not is_opacus_model:
        return state_dict
    
    cleaned_state_dict = {}
    for key, value in state_dict.items():
        # Remove '_module.' prefix added by Opacus
        if key.startswith('_module.'):
            new_key = key[8:]  # Remove '_module.' (8 characters)
            cleaned_state_dict[new_key] = value
        else:
            cleaned_state_dict[key] = value
    
    print(f"✅ Stripped Opacus prefixes from {len(cleaned_state_dict)} parameters")
    return cleaned_state_dict

# Load and evaluate the trained model
import os

# Use the model that was just trained in this notebook
model_path = "/root/advsecurenet_mp/examples/advsecurenet/defenses/adversarial_training/models/resnet18_cifar10_fgsm_adversarial_trained.pth"

if os.path.exists(model_path):
    # Load trained model
    checkpoint = torch.load(model_path, map_location='cpu')
    
    # Create fresh model for evaluation with CORRECT number of classes
    eval_model = ModelFactory.create_model(
        model_name="resnet18", 
        architecture={"num_classes": 10},  # Use architecture parameter
        pretrained=False
    )
    
    # Force the final layer to have exactly 10 classes (same as training)
    import torch.nn as nn
    
    if hasattr(eval_model, 'model'):
        base_model = eval_model.model
    else:
        base_model = eval_model
    
    # Force the final layer to have 10 classes
    if hasattr(base_model, 'fc'):
        in_features = base_model.fc.in_features
        base_model.fc = nn.Linear(in_features, 10)
    
    # FIX: Make model Opacus-compatible first
    from opacus.validators import ModuleValidator
    
    # Apply same fixes as training model
    def fix_inplace_operations(module):
        for name, child in module.named_children():
            if isinstance(child, torch.nn.ReLU):
                child.inplace = False
            else:
                fix_inplace_operations(child)

    if hasattr(eval_model, 'model'):
        fix_inplace_operations(eval_model.model)
    else:
        fix_inplace_operations(eval_model)
    
    eval_model = ModuleValidator.fix(eval_model)
    
    # Strip Opacus prefixes and load weights
    cleaned_state_dict = strip_opacus_prefix(checkpoint, is_opacus_model=True)
    
    try:
        eval_model.load_state_dict(cleaned_state_dict, strict=False)  # Use strict=False
        print(f"✅ Loaded model from {model_path} (with missing BN stats)")
    except Exception as e:
        print(f"⚠️ Model loading failed: {e}")
        print("Using current trainer model instead...")
        eval_model = trainer.model
    
    # Move to device and set to eval mode
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    eval_model = eval_model.to(device)
    eval_model.eval()
    
    print(f"📍 Model moved to: {device}")
    
else:
    print(f"❌ Model file not found: {model_path}")
    print("Using the current trained model for evaluation...")
    eval_model = trainer.model
    device = trainer._device

# Verify the evaluation model has correct architecture
if hasattr(eval_model, 'model') and hasattr(eval_model.model, 'fc'):
    fc_layer = eval_model.model.fc
elif hasattr(eval_model, 'fc'):
    fc_layer = eval_model.fc
else:
    fc_layer = None

if fc_layer:
    print(f"📊 Evaluation model final layer classes: {fc_layer.out_features}")
    if fc_layer.out_features == 10:
        print("✅ Evaluation model correctly configured for CIFAR-10")
    else:
        print(f"❌ Evaluation model has {fc_layer.out_features} classes instead of 10")

print(f"\n🔒 PRIVACY ANALYSIS:")
if differential_privacy_config.noise_multiplier == 0 or not differential_privacy_config.enable:
    print(f"   ⚠️  ACTUAL Privacy Level: NO PRIVACY (DP disabled or noise_multiplier = 0)")
    print(f"   📈 This is standard training, NOT differential privacy!")
    print(f"   🎯 Expected Accuracy: Normal (60-90%)")
else:
    print(f"   • Noise Multiplier: {differential_privacy_config.noise_multiplier}")
    print(f"   • Privacy Level: MODERATE TO STRONG")
    print(f"   • Expected Accuracy: ~30-70% (privacy-utility trade-off)")

In [ ]:
# Create test data loader and evaluate
test_loader = DataLoaderFactory.create_dataloader(
    dataset=test_data, 
    batch_size=256,
    shuffle=False,
    drop_last=False
)

# Evaluate the model
correct = 0
total = 0
class_correct = [0] * 10
class_total = [0] * 10

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

print("🔍 Evaluating trained ResNet-18 model on CIFAR-10 test set...")

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader):
        images, labels = images.to(device), labels.to(device)
        
        outputs = eval_model(images)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # Per-class accuracy
        c = (predicted == labels).squeeze()
        for i in range(labels.size(0)):
            label = labels[i]
            class_correct[label] += c[i].item()
            class_total[label] += 1
        
        if (batch_idx + 1) % 10 == 0:
            print(f"   Processed {batch_idx + 1}/{len(test_loader)} batches...")

# Overall accuracy
overall_accuracy = 100 * correct / total

print(f"\n🎯 CIFAR-10 TEST RESULTS:")
print(f"   📊 Test Accuracy: {overall_accuracy:.2f}% ({correct}/{total})")

if differential_privacy_config.enable and differential_privacy_config.noise_multiplier > 0:
    print(f"   🔒 Privacy Level: ε varies with noise_multiplier = {differential_privacy_config.noise_multiplier}")
    print(f"   📉 Privacy-Utility Trade-off Applied")
else:
    print(f"   📈 No Privacy Protection Applied (Standard Training)")

print(f"   🏆 Model Performance: {'Excellent' if overall_accuracy > 80 else 'Good' if overall_accuracy > 60 else 'Needs Improvement'}")

# Per-class breakdown
print(f"\n📋 Per-Class Accuracy Breakdown:")
print("   Class Name       | Accuracy")
print("   -----------------|----------")
for i in range(10):
    if class_total[i] > 0:
        class_acc = 100 * class_correct[i] / class_total[i]
        print(f"   {class_names[i]:15} | {class_acc:6.2f}%")

print(f"\n✅ Evaluation completed!")

In [ ]:
import torch
import os

def comprehensive_model_validation(model_path):
    """
    Comprehensive validation of model checkpoint structure.
    """
    if not os.path.exists(model_path):
        return {"error": f"File not found: {model_path}"}
    
    try:
        # Load checkpoint
        checkpoint = torch.load(model_path, map_location='cpu')
        
        info = {
            "file_exists": True,
            "file_size_mb": os.path.getsize(model_path) / (1024 * 1024),
            "checkpoint_type": type(checkpoint).__name__,
            "all_keys": list(checkpoint.keys()) if isinstance(checkpoint, dict) else [],
            "fc_layers": {},
            "prefix_analysis": {}
        }
        
        # Analyze all keys for prefixes
        all_keys = list(checkpoint.keys()) if isinstance(checkpoint, dict) else []
        
        # Count different prefix patterns
        prefixes = {"model.": 0, "_module.": 0, "_module.model.": 0, "no_prefix": 0}
        
        for key in all_keys:
            if key.startswith('_module.model.'):
                prefixes["_module.model."] += 1
            elif key.startswith('_module.'):
                prefixes["_module."] += 1
            elif key.startswith('model.'):
                prefixes["model."] += 1
            else:
                prefixes["no_prefix"] += 1
        
        info["prefix_analysis"] = prefixes
        
        # Find ALL fc-related keys
        fc_keys = [k for k in all_keys if 'fc' in k]
        
        print(f"\n🔍 ALL FC-RELATED KEYS in {os.path.basename(model_path)}:")
        for key in fc_keys:
            tensor = checkpoint[key]
            shape = list(tensor.shape) if hasattr(tensor, 'shape') else "No shape"
            print(f"   {key}: {shape}")
            
            info["fc_layers"][key] = {
                "shape": shape,
                "dtype": str(tensor.dtype) if hasattr(tensor, 'dtype') else "Unknown"
            }
        
        # Check for final layer specifically
        final_layer_candidates = [
            'fc.weight', 'model.fc.weight', '_module.fc.weight', 
            '_module.model.fc.weight', 'classifier.weight'
        ]
        
        detected_final_layer = None
        for candidate in final_layer_candidates:
            if candidate in checkpoint:
                detected_final_layer = candidate
                tensor = checkpoint[candidate]
                if hasattr(tensor, 'shape') and len(tensor.shape) == 2:
                    info["final_layer"] = {
                        "key": candidate,
                        "shape": list(tensor.shape),
                        "num_classes": tensor.shape[0],
                        "is_cifar10": tensor.shape[0] == 10,
                        "is_imagenet": tensor.shape[0] == 1000
                    }
                break
        
        return info
        
    except Exception as e:
        return {"error": f"Error loading checkpoint: {str(e)}"}

# Test all models
model_paths = [
    '/root/advsecurenet_mp/examples/advsecurenet/experiments/models/resnet18_cifar10_baseline.pth',
    './models/resnet18_cifar10_dp_no_noise_clipping_1.pth',
    './models/resnet18_cifar10_dp_average_privacy.pth'
]

print("🔍 COMPREHENSIVE MODEL VALIDATION:")
print("=" * 80)

for model_path in model_paths:
    print(f"\n📁 Model: {os.path.basename(model_path)}")
    print("-" * 50)
    
    info = comprehensive_model_validation(model_path)
    
    if "error" in info:
        print(f"   ❌ {info['error']}")
        continue
    
    print(f"   📏 File Size: {info['file_size_mb']:.2f} MB")
    print(f"   📋 Total Keys: {len(info['all_keys'])}")
    print(f"   🏷️  Checkpoint Type: {info['checkpoint_type']}")
    
    print(f"\n   🔑 PREFIX ANALYSIS:")
    for prefix, count in info["prefix_analysis"].items():
        if count > 0:
            print(f"      {prefix}: {count} keys")
    
    if "final_layer" in info:
        final = info["final_layer"]
        print(f"\n   🎯 FINAL LAYER DETECTED:")
        print(f"      Key: {final['key']}")
        print(f"      Shape: {final['shape']}")
        print(f"      Classes: {final['num_classes']}")
        
        if final['is_cifar10']:
            print(f"      ✅ CIFAR-10 Compatible (10 classes)")
        elif final['is_imagenet']:
            print(f"      🟠 ImageNet size (1000 classes)")
        else:
            print(f"      ❓ Unknown size ({final['num_classes']} classes)")
    else:
        print(f"\n   ❌ NO FINAL LAYER DETECTED")

print("\n" + "=" * 80)
print("✅ Validation complete!")